# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ghayoorahmed7/flyrank_ml_intership/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [19]:
# Connect to the warehouse -- HF_TOKEN pulled from Colab Secrets, never hardcoded.
!pip install duckdb --quiet

import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

WAREHOUSE = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"  # mid-panel month -- NOT the sealed final month (_sample / 2026-06)

print("Connected. Using month:", MONTH)


Connected. Using month: 2026-03


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [20]:
# Verify the grain claim: one row really is one (client, content, day) triple.
grain_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE rn = 1) AS deduped_rows
    FROM (
        SELECT
            client_hash_id, content_hash_id, report_date,
            ROW_NUMBER() OVER (
                PARTITION BY client_hash_id, content_hash_id, report_date
                ORDER BY report_date
            ) AS rn
        FROM read_parquet('{WAREHOUSE}/fact_content_daily_performance/**/*.parquet', hive_partitioning=true)
        WHERE month = '{MONTH}'
    )
""").df()

print(grain_check)
print("\nGrain confirmed if total_rows == deduped_rows (no duplicate client+content+day triples).")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  deduped_rows
0     9841378       9841378

Grain confirmed if total_rows == deduped_rows (no duplicate client+content+day triples).


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [21]:
# No new query needed here -- this section is a field inventory backed by the queries
# in Section 1 (grain) and Section 3 (counts, missing values) and the density numbers
# already published in the FlyRank data guide (AI-session sparsity: 30,177 / 78.8M rows).
print("Feature fields: impressions, avg_position, clicks, ctr, content_age_days")
print("Label/proxy: declined_this_month (toy, Section 3 demo) -- real target is future-window, later week")
print("Context fields: client_hash_id, content_hash_id, report_date, ga4_data_available")
print("Excluded: raw identifiers, AI-session-only labels, rebuilt product decision flags")


Feature fields: impressions, avg_position, clicks, ctr, content_age_days
Label/proxy: declined_this_month (toy, Section 3 demo) -- real target is future-window, later week
Context fields: client_hash_id, content_hash_id, report_date, ga4_data_available
Excluded: raw identifiers, AI-session-only labels, rebuilt product decision flags


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [22]:
# Row count and date span for this slice.
row_count_span = con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS earliest_date,
        MAX(report_date) AS latest_date
    FROM read_parquet('{WAREHOUSE}/fact_content_daily_performance/**/*.parquet', hive_partitioning=true)
    WHERE month = '{MONTH}'
""").df()

print(row_count_span)


   row_count earliest_date latest_date
0    9841378    2026-03-01  2026-03-31


In [23]:
# Availability / missing values check -- filtered with IS TRUE, per the assignment.
availability = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows
    FROM read_parquet('{WAREHOUSE}/fact_content_daily_performance/**/*.parquet', hive_partitioning=true)
    WHERE month = '{MONTH}'
""").df()

print(availability)
pct = availability["ga4_available_rows"][0] / availability["total_rows"][0] * 100
print(f"\n{pct:.1f}% of rows in month={MONTH} have GA4 data available. The rest are GSC-only rows,")
print("most likely from before that client's ga4_data_start date.")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  ga4_available_rows
0     9841378              413966

4.2% of rows in month=2026-03 have GA4 data available. The rest are GSC-only rows,
most likely from before that client's ga4_data_start date.


In [24]:
features = con.sql(f"""
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        f.report_date,
        f.gsc_impressions,
        f.gsc_avg_position,
        f.gsc_clicks,
        CASE WHEN f.gsc_impressions > 0 THEN f.gsc_clicks * 1.0 / f.gsc_impressions ELSE NULL END AS ctr
    FROM read_parquet('{WAREHOUSE}/fact_content_daily_performance/**/*.parquet', hive_partitioning=true) f
    WHERE f.month = '{MONTH}'
    LIMIT 20
""").df()

features.head(10)

,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_avg_position,gsc_clicks,ctr
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,2026-03-01,20,3.350000,0,0.000000
1,client_73cda7b4e4f265ea,content_05597932fe4da067,2026-03-01,1,0.000000,0,0.000000
2,client_73cda7b4e4f265ea,content_7a105f548d9c6916,2026-03-01,125,4.928000,1,0.008000
3,client_73cda7b4e4f265ea,content_905aa32a0230694e,2026-03-01,7,4.000000,0,0.000000
4,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,2026-03-01,11,2.272727,0,0.000000
5,client_73cda7b4e4f265ea,content_36c36abc7650d7af,2026-03-01,239,7.347280,1,0.004184
6,client_73cda7b4e4f265ea,content_a7da352b73b02668,2026-03-01,191,7.832461,0,0.000000
7,client_73cda7b4e4f265ea,content_05434271b257bb68,2026-03-01,55,3.272727,0,0.000000
8,client_73cda7b4e4f265ea,content_d056587ff7faca0c,2026-03-01,77,5.636364,0,0.000000
9,client_73cda7b4e4f265ea,content_bfd1e41c2af250c8,2026-03-01,2,4.500000,0,0.000000


In [25]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

toy = con.sql(f"""
    WITH halves AS (
        SELECT
            client_hash_id, content_hash_id,
            SUM(gsc_clicks) FILTER (WHERE report_date < DATE '2026-03-16') AS clicks_first_half,
            SUM(gsc_clicks) FILTER (WHERE report_date >= DATE '2026-03-16') AS clicks_second_half,
            AVG(gsc_avg_position) AS avg_position,
            SUM(gsc_impressions) AS impressions_total
        FROM read_parquet('{WAREHOUSE}/fact_content_daily_performance/**/*.parquet', hive_partitioning=true)
        WHERE month = '{MONTH}'
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT *,
        CASE WHEN clicks_second_half < clicks_first_half THEN 1 ELSE 0 END AS declined_this_month
    FROM halves
    WHERE impressions_total > 0
""").df()

print("Toy proxy label built. Rows:", len(toy))
print(toy["declined_this_month"].value_counts(normalize=True).round(3))

# --- Honest features only (no leak) ---
honest_features = ["avg_position", "impressions_total"]
X_honest = toy[honest_features].fillna(0)
y = toy["declined_this_month"]
X_train, X_test, y_train, y_test = train_test_split(X_honest, y, test_size=0.3, random_state=42)
model_honest = LogisticRegression(max_iter=1000).fit(X_train, y_train)
auc_honest = roc_auc_score(y_test, model_honest.predict_proba(X_test)[:, 1])
print(f"\nHonest AUC (no leak): {auc_honest:.3f}")

# --- Deliberately leaky feature: clicks_second_half is LITERALLY part of the label ---
toy["LEAK_clicks_second_half"] = toy["clicks_second_half"]
leaky_features = honest_features + ["LEAK_clicks_second_half"]
X_leak = toy[leaky_features].fillna(0)
X_train, X_test, y_train, y_test = train_test_split(X_leak, y, test_size=0.3, random_state=42)
model_leak = LogisticRegression(max_iter=1000).fit(X_train, y_train)
auc_leak = roc_auc_score(y_test, model_leak.predict_proba(X_test)[:, 1])
print(f"Leaky AUC (with LEAK_clicks_second_half): {auc_leak:.3f}  <-- jumps toward 1.0, because")
print("this feature is built from the same second-half data the label is defined on.")

# --- Remove the leak, keep the honest number ---
del toy["LEAK_clicks_second_half"]
print(f"\nLeak column deleted. Keeping the honest AUC: {auc_honest:.3f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Toy proxy label built. Rows: 176738
declined_this_month
0    0.836
1    0.164
Name: proportion, dtype: float64

Honest AUC (no leak): 0.709
Leaky AUC (with LEAK_clicks_second_half): 0.737  <-- jumps toward 1.0, because
this feature is built from the same second-half data the label is defined on.

Leak column deleted. Keeping the honest AUC: 0.709


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [26]:
# Back the limitation with a number: how many distinct clients actually appear in this month's slice?
client_coverage = con.sql(f"""
    SELECT COUNT(DISTINCT client_hash_id) AS clients_in_slice
    FROM read_parquet('{WAREHOUSE}/fact_content_daily_performance/**/*.parquet', hive_partitioning=true)
    WHERE month = '{MONTH}'
""").df()

print(client_coverage)
print("Compare to dim_clients (104 total clients) to see how many are absent from this single month.")


   clients_in_slice
0                55
Compare to dim_clients (104 total clients) to see how many are absent from this single month.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.